1. Counting customers in each order type. Lets the business see what order method is prefered and would allow them to invest more in that order method

In [3]:
SELECT COUNT(CustomerID) AS 'Number of Orders', 'Online Order' AS OrderType
FROM Sales.SalesOrderHeader
WHERE OnlineOrderFlag = 1
UNION
SELECT COUNT(CustomerID), 'In-Store Purchase' AS OrderType
FROM Sales.SalesOrderHeader
WHERE OnlineOrderFlag = 0;


(2 rows affected)

Total execution time: 00:00:00.033

Number of Orders,OrderType
27659,Online Order
3806,In-Store Purchase


2. Items that are in inventory but have never been ordered

In [14]:
SELECT p.ProductID, p.Name
FROM Production.Product p
JOIN Production.ProductInventory pi ON p.ProductID = pi.ProductID
EXCEPT
SELECT p.ProductID, p.Name
FROM Production.Product p
JOIN Sales.SalesOrderDetail sod ON p.ProductID = sod.ProductID;

(222 rows affected)

Total execution time: 00:00:00.069

ProductID,Name
1,Adjustable Race
2,Bearing Ball
3,BB Ball Bearing
4,Headset Ball Bearings
316,Blade
317,LL Crankarm
318,ML Crankarm
319,HL Crankarm
320,Chainring Bolts
321,Chainring Nut


3\. Vendors who never recieved orders

In [18]:
SELECT BusinessEntityID
FROM Purchasing.Vendor
EXCEPT
SELECT DISTINCT VendorID
FROM Purchasing.PurchaseOrderHeader;


(18 rows affected)

Total execution time: 00:00:00.012

BusinessEntityID
1596
1502
1642
1634
1670
1564
1528
1630
1532
1640


4. This Query lists all the customers who made a purchase with more than 1 sales persons

In [31]:
SELECT CustomerID, SalesPersonID
FROM Sales.SalesOrderHeader
WHERE SalesPersonID IS NOT NULL
INTERSECT
SELECT CustomerID, SalesPersonID
FROM Sales.SalesOrderHeader
GROUP BY CustomerID, SalesPersonID
HAVING COUNT(SalesPersonID) > 1
ORDER BY CustomerID ASC;

(729 rows affected)

Total execution time: 00:00:00.077

CustomerID,SalesPersonID
29484,279
29485,276
29486,275
29486,277
29487,275
29487,277
29488,286
29489,281
29490,283
29491,275


5\. This query shows a list of the sales employees and how many sales they did in what category. Helps to see who does well in what category

In [54]:
SELECT SalesPersonID, COUNT(SalesOrderID) AS "Number of Sales", 'ONLINE ORDER' AS "Order Type"
FROM Sales.SalesOrderHeader
WHERE SalesPersonID IS NOT NULL AND OnlineOrderFlag = 0
GROUP BY SalesPersonID 
UNION
SELECT SalesPersonID, COUNT(SalesOrderID) AS "Number of Sales", 'IN PERSON ORDER' AS "Order Type"
FROM Sales.SalesOrderHeader
WHERE OnlineOrderFlag = 1
GROUP BY SalesPersonID
ORDER BY SalesPersonID ASC;

(18 rows affected)

Total execution time: 00:00:00.047

SalesPersonID,Number of Sales,Order Type
NULL,27659,IN PERSON ORDER
274,48,ONLINE ORDER
275,450,ONLINE ORDER
276,418,ONLINE ORDER
277,473,ONLINE ORDER
278,234,ONLINE ORDER
279,429,ONLINE ORDER
280,95,ONLINE ORDER
281,242,ONLINE ORDER
282,271,ONLINE ORDER


6. Lists out orders that have a 'high value' to the business as well as the orders that have a 'low value' to the business. Really useful if cross referenced with items in those particular orders so you can see what items are valued by big spenders

In [60]:
SELECT SalesOrderID, CustomerID, TotalDue, 'High Value' AS OrderType
FROM Sales.SalesOrderHeader
WHERE TotalDue > 5000
UNION
SELECT SalesOrderID, CustomerID, TotalDue, 'Low Value' AS OrderType
FROM Sales.SalesOrderHeader
WHERE TotalDue < 100
ORDER BY CustomerID;



(13374 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.105

SalesOrderID,CustomerID,TotalDue,OrderType
54508,11012,82.8529,Low Value
68413,11012,6.9394,Low Value
74908,11013,82.8529,Low Value
56137,11013,43.0729,Low Value
54898,11014,43.0729,Low Value
53834,11019,59.659,Low Value
54332,11019,53.0069,Low Value
52626,11019,41.9569,Low Value
56910,11019,9.934,Low Value
57639,11019,8.0444,Low Value


7. Displays Products that have gotten a discount on the unit price

In [82]:
SELECT ProductID
FROM Production.Product
EXCEPT
SELECT DISTINCT ProductID
FROM Sales.SalesOrderDetail
WHERE UnitPriceDiscount > 0
Order BY ProductID ASC;


(504 rows affected)

Total execution time: 00:00:00.007

ProductID
1
2
3
4
316
317
318
319
320
321


8.  These are things we sell that we also use in production

In [85]:
SELECT ProductID
FROM Sales.SalesOrderDetail
INTERSECT
SELECT ComponentID
FROM Production.BillOfMaterials;


(218 rows affected)

Total execution time: 00:00:00.049

ProductID
925
902
733
756
779
802
971
825
948
919


9. Displays products that we either have in inventory or sold

In [87]:
SELECT DISTINCT p.ProductID, p.Name, 'Sold' AS Source
FROM Production.Product p
JOIN Sales.SalesOrderDetail as sod ON p.ProductID = sod.ProductID
UNION
SELECT DISTINCT p.ProductID, p.Name, 'Inventory' AS Source
FROM Production.Product p
JOIN Production.ProductInventory as pi ON p.ProductID = pi.ProductID;


(1008 rows affected)

Total execution time: 00:00:00.011

ProductID,Name,Source
1,Adjustable Race,Sold
879,All-Purpose Bike Stand,Sold
712,AWC Logo Cap,Sold
3,BB Ball Bearing,Sold
2,Bearing Ball,Sold
877,Bike Wash - Dissolver,Sold
316,Blade,Sold
843,Cable Lock,Sold
952,Chain,Sold
324,Chain Stays,Sold


10. Get the contact info of all customers, vendors and stores

In [92]:
SELECT BusinessEntityID, PhoneNumber, 'Customer' AS EntityType
FROM Person.PersonPhone
WHERE BusinessEntityID IN (SELECT CustomerID FROM Sales.Customer)
UNION
SELECT BusinessEntityID, PhoneNumber, 'Vendor' AS EntityType
FROM Person.PersonPhone
WHERE BusinessEntityID IN (SELECT BusinessEntityID FROM Purchasing.Vendor)
UNION
SELECT BusinessEntityID, PhoneNumber, 'Store' AS EntityType
FROM Person.PersonPhone
WHERE BusinessEntityID IN (SELECT BusinessEntityID FROM Sales.Store);


(10274 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.108

BusinessEntityID,PhoneNumber,EntityType
12181,1 (11) 500 555-0183,Customer
13133,258-555-0117,Customer
16940,409-555-0144,Customer
18634,826-555-0171,Customer
285,926-555-0182,Customer
11362,832-555-0115,Customer
11873,1 (11) 500 555-0114,Customer
16618,224-555-0191,Customer
17594,145-555-0182,Customer
18649,1 (11) 500 555-0140,Customer
